In [ ]:
from langgraph.graph import StateGraph
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from typing import TypedDict
from langgraph.constants import END, START
from pydantic import BaseModel, Field
from typing import Annotated, Literal
import operator

In [2]:
load_dotenv()

True

In [3]:
class Quadratic(TypedDict):

    a: int
    b: int
    c: int

    eq: str
    d: int
    result: str

In [4]:
def show_equation(state: Quadratic):

    eq = f"{state['a']}x2 + {state['b']}x + {state['c']}"
    return {'eq': eq}


def calculate_discriminant(state: Quadratic):

    d = state['b']**2 - 4*state['a']*state['c']
    return {'d': d}

def real_roots(state: Quadratic):

    root1 = (-state['b'] + (state['b']**2 - 4*state['a']*state['c'])**0.5) / 2*state['a']
    root2 = (-state['b'] - (state['b']**2 - 4*state['a']*state['c'])**0.5) / 2*state['a']

    result = f"The roots are {root1} and {root2}"
    return {'result': result}

def equal_roots(state: Quadratic):

    root = -state['b']/2*state['a']
    result =  f"The only real root is {root}"

    return {'result': result}

def imaginary_roots(state: Quadratic):

    result =  f"No real roots"
    return {'result': result}

def check_condition(state: Quadratic ) -> Literal['real_roots', 'equal_roots', 'imaginary_roots']:

    if state['d'] > 0:
        return "real_roots"
    elif state['d'] == 0:
        return "equal_roots"
    else:
        return "imaginary_roots"


In [5]:
graph = StateGraph(Quadratic)

graph.add_node("show_equation", show_equation)
graph.add_node("calculate_discriminant", calculate_discriminant)
graph.add_node("real_roots", real_roots)
graph.add_node("equal_roots", equal_roots)
graph.add_node("imaginary_roots", imaginary_roots)

graph.add_edge(START, "show_equation")
graph.add_edge("show_equation", "calculate_discriminant")

graph.add_conditional_edges(
    "calculate_discriminant",
    check_condition
)

graph.add_edge("real_roots", END)
graph.add_edge("equal_roots", END)
graph.add_edge("imaginary_roots", END)

workflow = graph.compile()

In [6]:
workflow.invoke({
    'a': 1,
    'b': 3,
    'c': 2
})

{'a': 1,
 'b': 3,
 'c': 2,
 'eq': '1x2 + 3x + 2',
 'd': 1,
 'result': 'The roots are -1.0 and -2.0'}